In [10]:
import pyarrow.parquet as pq
import duckdb
path = "/Users/anna/Desktop/Research/climate-investments/data/raw/builty_all.parquet"
pf = pq.ParquetFile(path)
print(pf.metadata)
#163,332,180 rows

  created_by: DuckDB version v1.4.4 (build 6ddac802ff)
  num_columns: 24
  num_rows: 163332180
  num_row_groups: 1329
  format_version: 1.0
  serialized_size: 5061079


In [11]:
#filter to residential only
input_file = path
residential_file = "/Users/anna/Desktop/Research/climate-investments/data/clean/temp/builty_residential.parquet"

duckdb.query(f"""
COPY (
    SELECT *
    FROM read_parquet('{input_file}')
    WHERE PROPERTY_TYPE = 'Residential'
)
TO '{residential_file}' (FORMAT PARQUET)
""")
df = pq.ParquetFile(residential_file)
print(df.metadata)
#60,211,670 rows when filtering to residential only

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  created_by: DuckDB version v1.4.4 (build 6ddac802ff)
  num_columns: 24
  num_rows: 60211670
  num_row_groups: 486
  format_version: 1.0
  serialized_size: 1932853


In [ ]:
#start elevation filtering
#key words to look for
STRICT_PATTERNS = [
    r"elevat",
    r"rais(e|ed|ing)",
    r"lift(ed|ing)",
    r"jack(ed|ing) up",
    r"flood",
    r"fema",
    r"bfe",
    r"base flood elevation",
    r"finished floor elevation",
    r"ffe",
    r"freeboard",
    r"nfip",
    r"icc",
    r"floodplain",
    r"flood plain",
    r"flood zone",
    r"hazard mitigation",
    r"substantial damage",
    r"substantially damaged",
    r"substantial improvement",
    r"substantially improved",
    r"storm surge",
    r"out of (the )?floodplain",
]
#key words to exclude
FALSE_POSITIVE_PATTERNS = [
    r"sewer|sanitary|septic",
    r"electrical permit",
    r"rewire",
    r"smoke detectors?",
    r"plumbing permit",
    r"mechanical permit",
    r"inspection",
    r"meter",
    r"pad",
    r"boat",
    r"fence",
    r"transfer",
    r"gas",
    r"a/c",
    r"new single family",
    r"new home",
    r"roof",
    r"window|windows",
    r"kitchen",
    r"bath",
    r"bedroom",
    r"outlet",
    r"wall",
    r"interior",
    r"demolish|demolition|demo",
    r"accessory",
    r"carport",
    r"porch",
    r"res - plumbing",
    r"res - mechanical",
    r"sheds/garage/",
    r"elevator",
    r"pool",
    r"mobile home",
    r"townhouse|town house",
    r"duplex",
    r"triplex",
    r"condo",
    r"elevator",
    r"guest house",
    r"solar|photovoltaic",
    r"signage|wall sign|channel letters|illuminated.{0,30}(sign|letter|cabinet)",
    r"tree removal|tree pruning|prun(e|ing)|canopy|oak tree|laurel oak",
    r"rais(e|ed|ing).{0,40}roof|roof.{0,40}rais(e|ed|ing)",
    r"rais(e|ed|ing).{0,40}ceiling|ceiling.{0,40}rais(e|ed|ing)",
    r"raise (the )?roof",
    r"raise (the )?bar",
    r"shower faucet|shower plumbing",
    r"generator",
    r"water heater",
    r"flood ?lights?",
    r"fire sprinklers?",
    r"suppression",
    r"fence permit|privacy fence|rock ?wall",
    r"plan elevations?",
    r"drawing",
    r"(north|side|front|rear|south|east|west|left|right).{0,30}elevations?",
    r"elevations?.{0,30}(north|side|front|rear|south|east|west|left|right)",
    r"elevation (drawing|plan|view|sheet|detail)",
    r"raised ranch",
    r"(raised|raise|raising).{0,40}curb|curb.{0,40}(raised|raise|raising)",
    r"patio",
]

FALSE_POSITIVE_EXCEPTION_PATTERNS = [
    r"raising for flood requirements",
    r"permit to elevate",
    r"house raising",
    r"fema house raising",
    r"raising existing house",
    r"elevate house",
    r"temp service.{0,80}elevate house",
    r"construction during house raising",
    r"power disconnected.{0,80}house raising",
    r"raised an existing mh",
    r"final elevation.{0,80}original slab.{0,80}raised portion",
    r"raise house",
    r"raise.{0,40}side of house",
    r"raise.{0,40}side of single family dwelling",
    r"raising the house",
    r"raising height of the building",
    r"lifting and moving house",
    r"elevating existing sfd",
    r"house.{0,40}(raised|lifted)",
    r"home.{0,40}(raised|lifted)",
    r"house is to be raised",
    r"house can be raised",
    r"house was raised",
    r"home was lifted",
    r"lifting home to level",
    r"raise home above flood level",
    r"house to be lifted",
    r"house lifting",
    r"raised house",
    r"raised home",
    r"house being moved",
    r"existing home to be moved",
    r"house raise",
    r"lifted up house",
    r"raise up existing single family house",
    r"raise sfd",
    r"raise exist\.? sfd",
    r"reconnect.{0,80}(plumbing|sewer|water|gas).{0,80}(raised|lifted|raise|lifting)",
    r"(plumbing|sewer|water|gas).{0,80}(reconnect|disconnect|cap).{0,80}(raise|raised|lift|lifted)",
    r"(disconnect|cap|reconnect).{0,80}(plumbing|sewer|water|gas).{0,80}(raise|raised|lift|lifted)",
    r"pipe dwv and water for house that was raised",
    r"water & drain work for raised home",
    r"raise existing structure",
    r"structure has been raised",
    r"structure will be raised",
    r"helical piles.{0,120}lift structure",
    r"pier and beam foundation.{0,120}south side.{0,40}raised",
    r"house raised between [0-9]+[-–][0-9]+ inches",
    r"foundation raise|raise foundation",
    r"foundation raise.{0,80}carport|carport.{0,80}foundation raise",
    r"foundation.{0,80}raise|raise.{0,80}foundation",
    r"raise.{0,80}install new foundation for existing",
    r"lift house",
    r"lifting of existing house",
    r"lifting the existing house",
    r"house.{0,80}will be lifted",
    r"moved house",
    r"raise existing s ?f ?r",
    r"raise existing single story (residence|home)",
    r"lift house above hightide",
    r"raise house off (of )?the foundation",
    r"raise off (of )?the foundation.{0,80}move house",
    r"stabilizing foundation.{0,80}lifting",
    r"set the house back down to existing elevation",
    r"elevat(e|ing).{0,40}existing s ?f res",
    r"rais(e|ing).{0,80}house.{0,120}windows?|lift(ing)? house.{0,120}windows?",
    r"elevat(e|ing).{0,40}(and remodel )?exi(st|sit)ing structure.{0,120}windows?",
    r"remodel & repair existing house.{0,120}raise house two feet|raise house two feet",
    r"elevation of living space",
]

LOOSE_FALSE_POSITIVE_PATTERNS = FALSE_POSITIVE_PATTERNS.copy()

In [13]:
'''
# False-positive patterns in build_builty_filter.py that are not exact matches
# to the current loose FALSE_POSITIVE_PATTERNS above.
STRICT_ONLY_FALSE_POSITIVE_PATTERNS = [
    r"flood damage repair",
    r"reconstruct|reconstruction|rebuild|new construction",
    r"mobile.{0,20}home.{0,20}move.{0,20}in",
    r"manufactured.{0,20}home.{0,20}move.{0,20}in",
    r"mobile.{0,20}home.{0,20}set.{0,20}up",
    r"elevation.{0,20}certificate|elevation.{0,20}assessor|elevation assessor's permit",
    r"model house|elevatoions",
    r"determination of substantial conformance.{0,120}elevations?",
    r"site modifications.{0,80}elevations?",
    r"townhouse|town house",
    r"elevation *#? *[0-9]+",
    r"(house type|model|master *file|masterfile).{0,100}elevation",
    r"elevation.{0,100}(house type|model|master *file|masterfile)",
    r"(build new|new).{0,80}(home|townhouse|town house|sfd|single family|custom home|residence).{0,100}elevation",
    r"(unit|lot) +[a-z0-9]+.{0,40}elevation",
    r"plan *#? *[0-9a-z-]+.{0,80}elevations?",
    r"elevations?.{0,80}(std gate|standard plan|master|tract|lot [0-9]+|nsfr|new sfr|new house)",
    r"(new sfr|new house|new single family|single family home|single family residential).{0,100}elevations?",
    r"elevations?.{0,100}(new sfr|new house|new single family|single family home|single family residential)",
    r"(front|rear|side|north|south|east|west|left|right).{0,30}elevations?",
    r"elevations?.{0,30}(front|rear|side|north|south|east|west|left|right)",
    r"(left|right) swing elevation",
    r"plan[: ]+[0-9a-z ]+.*elevation[: ]+[a-z]",
    r"elevation (drawing|plan|view|sheet|detail)",
    r"elevation ['\"]?[a-z][0-9]?['\"]?( |$|,|;|:|\\.)",
    r"elevations? ['\"]?[a-zivx]+[0-9]?['\"]?( |$|,|;|:|\\.)",
    r"['\"]?[a-z][0-9]?['\"]? elevation",
    r"repeat.{0,40}elevation",
    r"elevation[- ]+[a-z][0-9]?( |$|,|;|:|\\.)",
    r"(grade|grading|pad|site|curb|street|road|drain) elevation",
    r"raised slab|elevated slab",
    r"raised ranch",
    r"rais(e|ed|ing).{0,40}(porch|entry|garden|loop|meter|panel|wire|collar ties)",
    r"(porch|entry|garden|loop|meter|panel|wire|collar ties).{0,40}rais(e|ed|ing)",
    r"service.{0,40}rais(e|ed|ing)|rais(e|ed|ing).{0,40}service",
    r"back yard|front yard|side yard|rear yard",
    r"tree removal|tree pruning|prun(e|ing)|canopy|oak tree|laurel oak|roots.{0,60}(foundation|sidewalk|patio)",
    r"elevat(e|ed|ing).{0,60}(tree|canopy|limb|roof)|(?:tree|canopy|limb).{0,60}elevat(e|ed|ing)",
    r"sign.{0,80}house raising|house raising.{0,80}sign",
    r"code compliance.{0,100}house raising",
    r"signage|wall sign|channel letters",
    r"illuminated.{0,30}(sign|letter|cabinet)",
    r"finished floor",
    r"minimum ffe|minimun ffe|min ffe",
    r"elevation certificate",
    r"flood plain determination",
    r"patio addition rear elevation",
    r"front elevation refacing",
    r"new (home|sfr|single family|residence).{0,80}(plan |elevation [a-z]( |$))",
    r"(plan |master plan ).{0,30}elevation [a-z]( |$)",
    r"new (sfr|single family).{0,30}existing elevation",
    r"fence permit|rock ?wall|privacy fence",
]

print(f"Strict-only false-positive patterns: {len(STRICT_ONLY_FALSE_POSITIVE_PATTERNS)}")
'''

'\n# False-positive patterns in build_builty_filter.py that are not exact matches\n# to the current loose FALSE_POSITIVE_PATTERNS above.\nSTRICT_ONLY_FALSE_POSITIVE_PATTERNS = [\n    r"flood damage repair",\n    r"reconstruct|reconstruction|rebuild|new construction",\n    r"mobile.{0,20}home.{0,20}move.{0,20}in",\n    r"manufactured.{0,20}home.{0,20}move.{0,20}in",\n    r"mobile.{0,20}home.{0,20}set.{0,20}up",\n    r"elevation.{0,20}certificate|elevation.{0,20}assessor|elevation assessor\'s permit",\n    r"model house|elevatoions",\n    r"determination of substantial conformance.{0,120}elevations?",\n    r"site modifications.{0,80}elevations?",\n    r"townhouse|town house",\n    r"elevation *#? *[0-9]+",\n    r"(house type|model|master *file|masterfile).{0,100}elevation",\n    r"elevation.{0,100}(house type|model|master *file|masterfile)",\n    r"(build new|new).{0,80}(home|townhouse|town house|sfd|single family|custom home|residence).{0,100}elevation",\n    r"(unit|lot) +[a-z0-9]+.{0,

In [14]:
residential_file = "/Users/anna/Desktop/Research/climate-investments/data/clean/temp/builty_residential.parquet"
output_file = "/Users/anna/Desktop/Research/climate-investments/data/clean/temp/builty_loosefilter.parquet"

strict_sql = " OR ".join(
    [f"regexp_matches(desc_l, '{p}')" for p in STRICT_PATTERNS]
)

false_sql = " OR ".join(
    [f"regexp_matches(desc_l, '{p}')" for p in FALSE_POSITIVE_PATTERNS]
)

false_exception_sql = " OR ".join(
    [f"regexp_matches(desc_l, '{p}')" for p in FALSE_POSITIVE_EXCEPTION_PATTERNS]
)

false_sql = f"(({false_sql}) AND NOT ({false_exception_sql}))"

con = duckdb.connect()
con.execute("SET memory_limit='10GB'")
con.execute("SET threads=4")

query = f"""
COPY (
    WITH prepared AS (
        SELECT
            *,
            lower(coalesce(DESCRIPTION, '')) AS desc_l
        FROM read_parquet('{residential_file}')
    ),

    flagged AS (
        SELECT
            *,
            CASE WHEN ({strict_sql}) THEN 1 ELSE 0 END AS candidate_flag,
            CASE WHEN ({false_sql}) THEN 1 ELSE 0 END AS falsepos_flag
        FROM prepared
    )

    SELECT *
    FROM flagged
    WHERE candidate_flag = 1

) TO '{output_file}' (FORMAT PARQUET)
"""

con.execute(query)

summary = con.execute(f"""
    SELECT
        falsepos_flag,
        count(*) AS n,
        round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM read_parquet('{output_file}')
    GROUP BY falsepos_flag
    ORDER BY falsepos_flag
""").df()
print("Candidate-flagged rows by loose false-positive flag:")
display(summary)

state_summary = con.execute(f"""
    SELECT
        STATE,
        falsepos_flag,
        count(*) AS n
    FROM read_parquet('{output_file}')
    GROUP BY STATE, falsepos_flag
    ORDER BY STATE, falsepos_flag
""").df()
display(state_summary)

elevate1 = pq.ParquetFile(output_file)
print(elevate1.metadata)
# Keeps all candidate_flag == 1 rows, including falsepos_flag == 1, for diagnosis.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate-flagged rows by loose false-positive flag:


,falsepos_flag,n,pct
0,0,204508,23.51
1,1,665245,76.49


,STATE,falsepos_flag,n
0,AK,0,258
1,AK,1,629
2,AL,0,540
3,AL,1,1921
4,AR,0,195
...,...,...,...
95,WI,1,1882
96,WV,0,997
97,WV,1,5715
98,WY,0,24


  created_by: DuckDB version v1.4.4 (build 6ddac802ff)
  num_columns: 27
  num_rows: 869753
  num_row_groups: 8
  format_version: 1.0
  serialized_size: 42413


In [15]:
#export to stata
# read parquet
df = duckdb.query(f"""
SELECT *
FROM read_parquet('{output_file}')
""").to_df()

# write stata file
df.to_stata(
    "/Users/anna/Desktop/builty_loosefilter.dta",
    write_index=False,
    version=118
)


## Pipeline summary — observation counts

| Step | File | Obs | Notes |
|------|------|-----|-------|
| Raw | `data/raw/builty_all.parquet` | 163,332,180 | all property types |
| Residential filter | `data/clean/temp/builty_residential.parquet` | 60,211,670 | `PROPERTY_TYPE == 'Residential'` |
| Loose keyword filter | `data/clean/temp/builty_loosefilter.parquet` | rerun notebook | candidate=1; falsepos_flag kept as diagnostic |
| Export to Stata | `data/clean/temp/builty_loosefilter.dta` | rerun notebook | includes candidate rows regardless of falsepos_flag |
| **New records — genuine** | `data/clean/temp/builty_new_genuine.dta` | ~9,671 | `2_filter_new.do` |
| **Non-New records — filtered** | `data/clean/temp/builty_nonnew_filtered.dta` | ~9,800 | `3_filter_nonnew.do` |

### Stata do-files (next steps after this notebook)

**`2_filter_new.do`**  
Loads `builty_loosefilter.dta`, keeps `WORK_TYPES == "New"` (280,198 obs), drops "flood plain: no" boilerplate, applies genuine flood-adaptation signal requirement. Expected output: ~9,671 genuine new-construction flood-elevation cases.

**`3_filter_nonnew.do`**  
Loads `builty_loosefilter.dta`, drops `WORK_TYPES == "New"` (464,779 remain), applies boilerplate exclusions + additional non-New FP exclusions (raised deck/porch, flood lights, flood damage repair, no-elev-cert records, Zone X, HVAC same-elevation), then applies genuine signal requirement. Expected output: ~9,800 genuine non-New flood-adaptation cases.

### Why so few survive?
The loose filter's main false positive drivers:
- `elevat` → "plan elevation: A/B/C" builder façade labels (new construction)
- `flood` → "flood plain: no" standard checklist boilerplate on all new builds
- `rais` → raised decks, patio additions, electrical mast raising
- `elevat` + `flood` in same record but far apart / unrelated context

In [16]:
#check tx and va remaining amount
duckdb.query(f"""
SELECT STATE, COUNT(*) AS n
FROM read_parquet('{output_file}')
WHERE upper(STATE) IN ('TX', 'VA')
GROUP BY STATE
ORDER BY STATE
""").to_df()

,STATE,n
0,TX,75029
1,VA,54025


---
## Filter investigation

Applies the strict patterns from `build_builty_filter.py` to the loosefilter (TX+VA) and keeps **all rows** with diagnostic flags, so you can sample each population:

| Flag | Meaning |
|---|---|
| `flood_elev_strict` | matched a strict home-elevation pattern |
| `flood_adaptation_context` | matched a flood/FEMA/BFE context pattern |
| `flood_elev_falsepos` | matched a false-positive exclusion pattern |
| `flood_elev_final` | strict=1 AND falsepos=0 → **final kept** |

**Populations to check:**
1. `flood_elev_final == 1` → kept — do these look genuine?
2. `strict == 1, falsepos == 1` → FP-dropped — are any of these real elevations?
3. `strict == 0` → missed — are any of these genuine that the strict patterns are missing?
4. `strict == 1, context == 0, falsepos == 0` → kept but no flood context — noise risk?

In [17]:
import sys
import pandas as pd

code_root        = "/Users/anna/Desktop/climate-investments/code"
data_root        = "/Users/anna/Desktop/Research/climate-investments/data"
loosefilter_path = f"{data_root}/clean/temp/builty_loosefilter.parquet"

# import patterns + SQL helpers from build_builty_filter.py so investigation stays in sync
sys.path.insert(0, f"{code_root}/build")
from build_builty_filter import (
    STRICT_PATTERNS, FLOOD_CONTEXT_PATTERNS,
    FALSE_POSITIVE_PATTERNS as STRICT_FALSE_POSITIVE_PATTERNS,
    regex_any_sql, quote_sql,
)

strict_fp_set = set(STRICT_FALSE_POSITIVE_PATTERNS)
loose_fp_set = set(LOOSE_FALSE_POSITIVE_PATTERNS)

print(f"Loose false-positive patterns here: {len(loose_fp_set)}")
print(f"build_builty_filter.py false-positive patterns: {len(strict_fp_set)}")
print(f"Shared exact patterns: {len(loose_fp_set & strict_fp_set)}")
print(f"Only in loose filter here: {len(loose_fp_set - strict_fp_set)}")
print(f"Only in build_builty_filter.py: {len(strict_fp_set - loose_fp_set)}")

fp_pattern_comparison = pd.concat([
    pd.DataFrame({"where": "only_here_loose_filter", "pattern": sorted(loose_fp_set - strict_fp_set)}),
    pd.DataFrame({"where": "only_build_builty_filter", "pattern": sorted(strict_fp_set - loose_fp_set)}),
    pd.DataFrame({"where": "shared_exact", "pattern": sorted(loose_fp_set & strict_fp_set)}),
], ignore_index=True)

display(fp_pattern_comparison)

Loose false-positive patterns here: 60
build_builty_filter.py false-positive patterns: 64
Shared exact patterns: 14
Only in loose filter here: 46
Only in build_builty_filter.py: 50


,where,pattern
0,only_here_loose_filter,(north|side|front|rear|south|east|west|left|ri...
1,only_here_loose_filter,"(raised|raise|raising).{0,40}curb|curb.{0,40}(..."
2,only_here_loose_filter,a/c
3,only_here_loose_filter,accessory
4,only_here_loose_filter,bath
...,...,...
105,shared_exact,raised ranch
106,shared_exact,shower faucet|shower plumbing
107,shared_exact,townhouse|town house
108,shared_exact,triplex


In [18]:
strict_sql   = regex_any_sql("desc_l", STRICT_PATTERNS)
context_sql  = regex_any_sql("desc_l", FLOOD_CONTEXT_PATTERNS)
falsepos_sql = regex_any_sql("desc_l", STRICT_FALSE_POSITIVE_PATTERNS)

con = duckdb.connect()
con.execute("SET memory_limit='10GB'")
con.execute("SET threads=4")

df = con.execute(f"""
    WITH prepared AS (
        SELECT
            * EXCLUDE(desc_l),
            lower(coalesce(DESCRIPTION, '')) AS desc_l
        FROM read_parquet({quote_sql(loosefilter_path)})
        WHERE upper(STATE) IN ('TX', 'VA')
    ),
    flagged AS (
        SELECT *,
            CASE WHEN ({strict_sql})   THEN 1 ELSE 0 END AS flood_elev_strict,
            CASE WHEN ({context_sql})  THEN 1 ELSE 0 END AS flood_adaptation_context,
            CASE WHEN ({falsepos_sql}) THEN 1 ELSE 0 END AS flood_elev_falsepos
        FROM prepared
    )
    SELECT *,
        CASE WHEN flood_elev_strict = 1 AND flood_elev_falsepos = 0 THEN 1 ELSE 0 END AS flood_elev_final
    FROM flagged
""").df()

print(f"Loaded {len(df):,} TX+VA records from loosefilter")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 129,054 TX+VA records from loosefilter


In [19]:
funnel = pd.DataFrame({
    "stage": [
        "loosefilter (TX+VA)",
        "strict text hit",
        "strict hit — FP excluded",
        "strict hit — no flood context",
        "FINAL KEPT",
    ],
    "n": [
        len(df),
        int((df.flood_elev_strict == 1).sum()),
        int(((df.flood_elev_strict == 1) & (df.flood_elev_falsepos == 1)).sum()),
        int(((df.flood_elev_strict == 1) & (df.flood_adaptation_context == 0) & (df.flood_elev_falsepos == 0)).sum()),
        int((df.flood_elev_final == 1).sum()),
    ]
})
funnel["pct_of_loosefilter"] = (funnel.n / len(df) * 100).round(1).astype(str) + "%"
display(funnel)

,stage,n,pct_of_loosefilter
0,loosefilter (TX+VA),129054,100.0%
1,strict text hit,3873,3.0%
2,strict hit — FP excluded,3452,2.7%
3,strict hit — no flood context,242,0.2%
4,FINAL KEPT,421,0.3%


In [20]:
pd.set_option('display.max_colwidth', 400)

def show(pop, n=20, seed=42):
    """Sample n rows and display DESCRIPTION + key columns."""
    cols = [c for c in ['STATE', 'WORK_TYPES', 'DESCRIPTION', 'desc_l'] if c in pop.columns]
    return pop[cols].sample(min(n, len(pop)), random_state=seed).reset_index(drop=True)

In [21]:
# --- Population 1: kept (final output) ---
# Check: do these look like genuine flood elevation permits?
kept = df[df.flood_elev_final == 1]
print(f"Kept: {len(kept):,}")
show(kept)

Kept: 421


,STATE,WORK_TYPES,DESCRIPTION,desc_l
0,TX,Alteration,Residential Foundation Repair Pre-3.12.21\nInstall 10 Piers around the base of the foundation to raise the home,residential foundation repair pre-3.12.21\ninstall 10 piers around the base of the foundation to raise the home
1,VA,NaN,Plumbing Residential\nReconnecting water and sewer under house due to house being raised in flood zone,plumbing residential\nreconnecting water and sewer under house due to house being raised in flood zone
2,TX,NaN,"Development Permit - Floodplain ($110)\nEarthworks within 100-year floodplain to raise home foundation and install levee -based retention pond to offset home pad fill placed in 100-year floodplain.\nFlood mitigation plan submitted on 20 AUG 2019, with Ft. Bend County responding with 'Letter","development permit - floodplain ($110)\nearthworks within 100-year floodplain to raise home foundation and install levee -based retention pond to offset home pad fill placed in 100-year floodplain.\nflood mitigation plan submitted on 20 aug 2019, with ft. bend county responding with 'letter"
3,VA,New,Residential Foundation Only\nThe house is being raised so that the location/type of the current piers and other foundation can be determined. The new foundation is not being done at this time. A new permit will be submitted with all relevant documentation prior to the new foundation being started.,residential foundation only\nthe house is being raised so that the location/type of the current piers and other foundation can be determined. the new foundation is not being done at this time. a new permit will be submitted with all relevant documentation prior to the new foundation being started.
4,VA,Alteration\nNew,Electrical Residential\nINSTALL NEW WIRING FOR RENOVATION OF GARAGE AFTER HOUSE RAISING\nASSOCIATED PERMIT 2019-BDRA-11088,electrical residential\ninstall new wiring for renovation of garage after house raising\nassociated permit 2019-bdra-11088
5,TX,Alteration\nDemolition\nNew,Building Permit\nRemodel to repair home to include: repair rotted siding repair sheetrock & repair plumbing leaks in master bath & pipe under house near kitchen; a study floor will be raised to the same level as the rest of the home and study windows raised to accomodate higher floor height remove interior wall between kitchen and dining rm install french /double doors between living rm & stud...,building permit\nremodel to repair home to include: repair rotted siding repair sheetrock & repair plumbing leaks in master bath & pipe under house near kitchen; a study floor will be raised to the same level as the rest of the home and study windows raised to accomodate higher floor height remove interior wall between kitchen and dining rm install french /double doors between living rm & stud...
6,TX,New,Residential Building Permit\nraising house,residential building permit\nraising house
7,TX,Alteration,Electrical Permit\ndisconnect & reconnect for grounding after house was raised,electrical permit\ndisconnect & reconnect for grounding after house was raised
8,TX,NaN,"FOUNDATION-SLAB\nRAISE HOUSE 24"" PER ATTACHED","foundation-slab\nraise house 24"" per attached"
9,TX,Addition\nAlteration\nDemolition\nNew,Plumbing Permit\nPartial demo to remove part of N wall where porch is front part of W wall shed in back & front existing carport. Remodel to remove non load bearing closet interior walls and add some interior walls new HVAC new plumbing and electrical throughout house raise floor of enclosed garage raise corresponding exteriror walls replace windows and doors. Addition to add a carport (NW sid...,plumbing permit\npartial demo to remove part of n wall where porch is front part of w wall shed in back & front existing carport. remodel to remove non load bearing closet interior walls and add some interior walls new hvac new plumbing and electrical throughout house raise floor of enclosed garage raise corresponding exteriror walls replace windows and doors. addition to add a carp

In [22]:
# --- Population 2: FP-dropped ---
# Strict pattern matched but a false-positive pattern also matched.
# Check: is the FP pattern too broad and catching genuine elevations?
fp_dropped = df[(df.flood_elev_strict == 1) & (df.flood_elev_falsepos == 1)]
print(f"FP-dropped: {len(fp_dropped):,}  ← strict hit excluded by a false-positive pattern")
show(fp_dropped)

FP-dropped: 3,452  ← strict hit excluded by a false-positive pattern


,STATE,WORK_TYPES,DESCRIPTION,desc_l
0,TX,New,"Building Permits (Construction, Electrical, Plumbing, HVAC, etc.) - Home Elevation Permit","building permits (construction, electrical, plumbing, hvac, etc.) - home elevation permit"
1,TX,Alteration\nNew,Building Pmt\nRESIDENTIAL HOME ELEVATION,building pmt\nresidential home elevation
2,TX,New,Electrical Permit\nBlanton 1610 - Right Swing Elevation P New 1 Story SFR home with (3) bdrms (2) bath 2-car garage covered patio covered entry porch,electrical permit\nblanton 1610 - right swing elevation p new 1 story sfr home with (3) bdrms (2) bath 2-car garage covered patio covered entry porch
3,TX,Alteration,CRT/COMPLIANCE\nRESIDENTIAL HOME ELEVATION,crt/compliance\nresidential home elevation
4,TX,Addition\nNew,Sub Permit - Electrical\nNew Single Family House - PLAN 2385 ELEVATION V (LH)\n*****************************************\nFollow all City Policies and Standard Operating Procedures.\nAll construction must comply with the City of League Code adopted codes in addition to state adopted codes.\nThe City has adopted the 2015 I-Codes and 2020 NEC.\nWind load/windstorm must be rated for the ultimate ...,sub permit - electrical\nnew single family house - plan 2385 elevation v (lh)\n*****************************************\nfollow all city policies and standard operating procedures.\nall construction must comply with the city of league code adopted codes in addition to state adopted codes.\nthe city has adopted the 2015 i-codes and 2020 nec.\nwind load/windstorm must be rated for the ultimate ...
5,TX,Addition\nNew,SFR Sub Mechanical\nMechanical Service - new home; repeat 428 SFR210113\nNEW SFR - PLAN 428 ELEVATION A (LH)\n*****************************************\nFollow all City Policies and Standard Operating Procedures.\nConstruction staging and material storage is prohibited in City Right-of-Way and/or Easements.\nAll construction must comply with the City of League Code adopted codes in addition to...,sfr sub mechanical\nmechanical service - new home; repeat 428 sfr210113\nnew sfr - plan 428 elevation a (lh)\n*****************************************\nfollow all city policies and standard operating procedures.\nconstruction staging and material storage is prohibited in city right-of-way and/or easements.\nall construction must comply with the city of league code adopted codes in addition to...
6,TX,Alteration,Drainage Review\nRESIDENTIAL FLOOD DAMAGE HOUSE ELEVATION,drainage review\nresidential flood damage house elevation
7,TX,Alteration,Plumbing Permit\nWhole house gas pressure test for pressure elevation request,plumbing permit\nwhole house gas pressure test for pressure elevation request
8,TX,NaN,Electrical Permit\nWall sign The Home Depot east elevation (Re-permit of 2008-054182),electrical permit\nwall sign the home depot east elevation (re-permit of 2008-054182)
9,VA,Addition\nAlteration\nNew,Building Addition and or Alteration Residential\n22x37 inground pool POOL EQUIPMENT MUST BE OUT OF FLOOD ZONE OR ELEVATED ABOVE BFE,building addition and or alteration residential\n22x37 inground pool pool equipment must be out of flood zone or elevated above bfe


In [23]:
# --- Population 3: no strict hit ---
# These passed the loose filter but not the strict one.
# Check: are any of these genuine elevations the strict patterns are missing?
missed = df[df.flood_elev_strict == 0]
print(f"No strict hit: {len(missed):,}  ← in loosefilter but not caught by strict patterns")
show(missed)

No strict hit: 125,181  ← in loosefilter but not caught by strict patterns


,STATE,WORK_TYPES,DESCRIPTION,desc_l
0,TX,New,Mechanical Permit\nHarmony - 2253-G-R 2 story single family residence with 4 bedrooms 2.5 bathrooms attached 2 car garage covered entry and patio. Building Height: 25-11 Maximum Height: 35 Required Parking: 2. Required FFE:678.3 - above MSL. Site Plan#: SP-2020-0091C. Expiration Date: 06-15-2024.,mechanical permit\nharmony - 2253-g-r 2 story single family residence with 4 bedrooms 2.5 bathrooms attached 2 car garage covered entry and patio. building height: 25-11 maximum height: 35 required parking: 2. required ffe:678.3 - above msl. site plan#: sp-2020-0091c. expiration date: 06-15-2024.
1,TX,New,Design Review Fees; On-Site Sewage Facility (SFR); Temporary - Floodplain Development Permit (SFR)\nNEW SINGLE FAMILY RESIDENCE & OSSF,design review fees; on-site sewage facility (sfr); temporary - floodplain development permit (sfr)\nnew single family residence & ossf
2,TX,Addition\nNew,Electrical Permit\n2380 Bruckner- E2 Right Single Family 2-Story Condominium4-Bdrm 3 Bath 2-Car Garage. With additional info of: Building Height: 28-3 Maximum Height: 32. Required Parking: 2. Required FFE: 898.46,electrical permit\n2380 bruckner- e2 right single family 2-story condominium4-bdrm 3 bath 2-car garage. with additional info of: building height: 28-3 maximum height: 32. required parking: 2. required ffe: 898.46
3,TX,NaN,Floodplain Development Permit (SFR)\nMOBILE HOME,floodplain development permit (sfr)\nmobile home
4,TX,NaN,Impact Fees - Impact Fees Effective 06/01/2021\n101- SINGLE FAMILY HOUSES- DETACHED HATTON PLACE L 438,impact fees - impact fees effective 06/01/2021\n101- single family houses- detached hatton place l 438
5,VA,New,"NEW TOWNHOUSE\ntownhouse/ansi/ KIRKTON / with elevation 1, end unit, 2 car garage, elevator shaft, ext rear terrace and master suite ext.","new townhouse\ntownhouse/ansi/ kirkton / with elevation 1, end unit, 2 car garage, elevator shaft, ext rear terrace and master suite ext."
6,TX,NaN,Temporary - Floodplain Development Permit (SFR)\nSINGLE FAMILY RESIDENCE,temporary - floodplain development permit (sfr)\nsingle family residence
7,TX,New,Plumbing Permit\nNew 3-story Condominium Residence with attached garage. **CONDOMINIUM RESIDENCE**\n**Revision at 3rd floor bathroom revision at all exterior elevations and all corresponding structural drawings**,plumbing permit\nnew 3-story condominium residence with attached garage. **condominium residence**\n**revision at 3rd floor bathroom revision at all exterior elevations and all corresponding structural drawings**
8,VA,NaN,Residential Driveway Entrance Permit - BLD\nSF DWELLING PORCH & DECK - DEFERRED PROFFERS,residential driveway entrance permit - bld\nsf dwelling porch & deck - deferred proffers
9,TX,Alteration,Residential Alterations/Repairs Permit (Effective 10/01/2024)\nTEAR OFF AND REROOF,residential alterations/repairs permit (effective 10/01/2024)\ntear off and reroof


In [24]:
# --- Population 4: strict hit, no flood context ---
# Kept by default in this diagnostic population. Review before tightening context requirements.
# Check: are these genuine elevations that just lack flood keywords, or noise?
no_context = df[
    (df.flood_elev_strict == 1) &
    (df.flood_adaptation_context == 0) &
    (df.flood_elev_falsepos == 0)
]
print(f"Strict hit, no flood context: {len(no_context):,}")
show(no_context)

Strict hit, no flood context: 242


,STATE,WORK_TYPES,DESCRIPTION,desc_l
0,TX,Addition\nNew,"New Single Family Residential (Effective 3/1/2023); Storage Building (Effective 3/1/2023)\nSFR Addition\nAdding 12x12 shed on foundation, and refurbishing an un-enclosed existing lean-to on side of house (including raising the height)","new single family residential (effective 3/1/2023); storage building (effective 3/1/2023)\nsfr addition\nadding 12x12 shed on foundation, and refurbishing an un-enclosed existing lean-to on side of house (including raising the height)"
1,TX,NaN,RESIDENTIAL\nlevel & stabilize foundation -- 11 PIERS NO LIFTING OF HOUSE,residential\nlevel & stabilize foundation -- 11 piers no lifting of house
2,TX,New,Legacy - Permit\nFoundation/Leveling elevating home 4.5 feet according to engineered designs. To including building of engineered steps,legacy - permit\nfoundation/leveling elevating home 4.5 feet according to engineered designs. to including building of engineered steps
3,VA,Addition\nAlteration\nDemolition\nNew,"Plumbing Residential\nAT OWNERS RISK PENDING VARIANCE. WILL BE LIFTING EXISTING SINGLE STORY STRUCTURE, CONSTRUCT NEW 1ST STORY AND 3RD LEVEL UNFINISHED LOFT/STORAGE AREA.\nPERMIT TO INCLUDE FOLLOWING PENDING VARIANCE APPROVAL:\n- E&S CONTROLS SITE WORK, FOOTING/FOUNDATIONS FOR ALL LATERAL ADDITIONS, INTERIOR DEMO AND REMODEL WITH STRUCTURAL WORK NECESSARY, WINDOWS/DOORS.\n**11/8/16 REV VARIAN...","plumbing residential\nat owners risk pending variance. will be lifting existing single story structure, construct new 1st story and 3rd level unfinished loft/storage area.\npermit to include following pending variance approval:\n- e&s controls site work, footing/foundations for all lateral additions, interior demo and remodel with structural work necessary, windows/doors.\n**11/8/16 rev varian..."
4,VA,NaN,"PPR Standard\nLot 12, Lynnhaven Colony\nRaise the house per specifications of the SRL Grant Program. No change in footprint.","ppr standard\nlot 12, lynnhaven colony\nraise the house per specifications of the srl grant program. no change in footprint."
5,VA,Addition\nAlteration\nNew,Building Addition and or Alteration Residential\nDECK\n**STAIRS TO FRONT AND REAR ENTRY OF PROPERTY FROM LIFTING THE HOUSE **\n**OK BY COLE FISHER **,building addition and or alteration residential\ndeck\n**stairs to front and rear entry of property from lifting the house **\n**ok by cole fisher **
6,VA,Addition\nAlteration,"Mechanical Residential Alteration & Repair\nRAISING HOUSE 2 FT IN ORDER TO REACH 7.8FT HEIGHT AND FINISHING BASEMENT AS HABITABLE SPACE - *REVISED TO ADD REVOVATION OF MAIN LEVEL OF HOME - FIX/REPAIR DRYWALL/ELECTRICAL, REPLACE APPLIANCES, ETC. - 1/14/2020*","mechanical residential alteration & repair\nraising house 2 ft in order to reach 7.8ft height and finishing basement as habitable space - *revised to add revovation of main level of home - fix/repair drywall/electrical, replace appliances, etc. - 1/14/2020*"
7,TX,NaN,Legacy - Permit\nElevate home according to plans.,legacy - permit\nelevate home according to plans.
8,TX,Alteration\nDemolition\nNew,"Accessory Structure\n1806 Keneipp Rd, Foundation 2. *Inspect the entire house 3. *Raise house (with jacks), up to 12"" as needed 4. *Cover pillars using 2"" ×4"" as needed 5. *install new 17 piers 6. *Reinforce pillars with corner brackets, around pillars 7. *Reinforce floor joist/columns to pillars 8. *Lower house back into place and remove jacks","accessory structure\n1806 keneipp rd, foundation 2. *inspect the entire house 3. *raise house (with jacks), up to 12"" as needed 4. *cover pillars using 2"" ×4"" as needed 5. *install new 17 piers 6. *reinforce pillars with corner brackets, around pillars 7. *reinforce floor joist/columns to pillars 8. *lower house back into place and remove jacks"
9,TX,Addition\nAlteration,CRT/COMPLIANCE\nRESIDENTIAL HOUSE RAISE AND REMODEL/ADDI,crt/compliance\nresidential house raise and remodel/addi
